In [0]:
# Define raw transaction data inline (simulating a CSV source)
raw_data = [
    {"transaction_id": "TXN001", "customer_id": "CUST01", "amount": 150.00, "currency": "USD", "transaction_date": "2024-01-15", "merchant": "Amazon"},
    {"transaction_id": "TXN002", "customer_id": "CUST02", "amount": 89.99, "currency": "USD", "transaction_date": "2024-01-15", "merchant": "Walmart"},
    {"transaction_id": "TXN003", "customer_id": "CUST01", "amount": -20.00, "currency": "USD", "transaction_date": "2024-01-16", "merchant": "Refund Co"},
    {"transaction_id": "TXN004", "customer_id": "CUST03", "amount": 250.50, "currency": "EUR", "transaction_date": "2024-01-16", "merchant": "Shopify"},
    {"transaction_id": "TXN005", "customer_id": None, "amount": 75.00, "currency": "USD", "transaction_date": "2024-01-17", "merchant": "Target"},
    {"transaction_id": None, "customer_id": "CUST04", "amount": 120.00, "currency": "GBP", "transaction_date": "2024-01-17", "merchant": "Tesco"},
]



In [0]:
# Create a PySpark DataFrame from the raw data
df = spark.createDataFrame(raw_data)

In [0]:
display(df)

In [0]:
# Write raw data as-is to the bronze Delta Lake table
# Delta Lake is the default format - .format("delta") is explicit but optional
df.write.format("delta").mode("append").saveAsTable("workspace.default.bronze_transactions")

In [0]:
%sql
-- Verify the bronze table contains the ingested rows
SELECT * FROM default.bronze_transactions

In [0]:
# Define a second batch of transactions with a new 'category' field
new_data = [
    ("TXN006", "C003", 250.00, "USD", "2024-01-06", "TechStore", "electronics"),
    ("TXN007", "C001", 45.00, "USD", "2024-01-07", "GroceryMart", "groceries"),
    ("TXN008", "C004", 180.00, "EUR", "2024-01-08", "FashionHub", "clothing"),
]

# Column names now include the new 'category' field
new_columns = ["transaction_id", "customer_id", "amount", "currency", "transaction_date", "merchant", "category"]

# Create a DataFrame from the new batch
new_df = spark.createDataFrame(new_data, new_columns)

# Write to the bronze table with schema evolution enabled
new_df.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.bronze_transactions")

print("Second batch written to bronze with schema evolution!")

In [0]:
# Verify the bronze table schema has evolved to include 'category'
result = spark.sql("SELECT * FROM workspace.default.bronze_transactions")
result.display()